# Downloading articles using Wikidumps

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

## Fetch QIDs and titels
  
Use the wikidata entity dump to fetch QIDs and titels  
only fetch the data that contains the required 5 languages  
1000 QIDs * 5 languages = 5000 articles

In [ ]:
import requests
import bz2
import json

# URL of the compressed dump
url = "https://dumps.wikimedia.org/wikidatawiki/entities/latest-all.json.bz2"

# Fetch the compressed data in chunks
response = requests.get(url, stream=True)

# Create a decompressor instance for bz2 data
decompressed = bz2.BZ2Decompressor()

# Define the languages we're interested in
lang_keys = ['enwiki', 'dewiki', 'eswiki', 'zhwiki', 'hiwiki']

# Initialize counters and buffers
count = 0
buffer = b""
entities = []

# Loop through chunks of data from the response
for chunk in response.iter_content(4096):
    buffer += decompressed.decompress(chunk)

    # Process each line when it's complete (lines are separated by newline characters)
    while b"\n" in buffer:
        line, buffer = buffer.split(b"\n", 1)
        try:
            line = line.decode("utf-8").strip().rstrip(',')
            if line.startswith('{'):
                obj = json.loads(line)
                sitelinks = obj.get("sitelinks", {})

                # Check if all required language sitelinks are available
                if all(lang in sitelinks for lang in lang_keys):
                    # Extract the required information and append it to the entities list
                    entity = {
                        "qid": obj["id"],
                        **{lang: sitelinks[lang]["title"] for lang in lang_keys}
                    }
                    entities.append(entity)

                    count += 1
                    if count >= 1000:
                        print("Reached 1000 entities. Stopping.")
                        break  # Exit the while loop if processed the desired number of articles
        except Exception:
            # If there is an error in processing (e.g., invalid line), skip to the next line
            continue

    # Break the outer loop if the limit is reached
    if count >= 1000:
        break

## Extract articles
Use Wikipedia API to extract articles based on the QID and titles

In [ ]:
import requests
import time
import csv


# File setup
output_file = '/content/drive/My Drive/wikipedia/wiki_extracts.csv'
fieldnames = ['qid', 'language', 'title', 'extract']

# Wikipedia API fetch function
def fetch_extract(lang_code, title):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "format": "json",
        "titles": title,
        "explaintext": 1
    }
    headers = {
        "User-Agent": "WikiArticlesExtractor/1.0 (mailto:huan.liu2@uzh.ch)"
    }
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        page = next(iter(data["query"]["pages"].values()))
        return page.get("extract", "")
    except Exception as e:
        return f"ERROR: {e}"

## Clean the extracted articles
Remove the markup and references

In [ ]:
import re

def clean_wikipedia_text(text):
    # Remove section headers (markup)
    text = re.sub(r'==+[^=]+==+', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Stop at the References or See also section
    for section in ['References', 'See also', 'External links', 'Footnotes', 'Bibliography']:
        text = re.split(rf'\b{section}\b', text, flags=re.IGNORECASE)[0]

    return text.strip()

## Extract and save the data
Save the extracted data as a csv file.   

In [ ]:
from tqdm import tqdm
import time
import csv

# Write to CSV
with open(output_file, mode='w', encoding='utf-8', newline='') as file:
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    for entity in tqdm(entities, desc="Processing entities"):
        for lang_key in ['enwiki', 'dewiki', 'eswiki', 'zhwiki', 'hiwiki']:
            lang_code = lang_key[:2]
            title = entity[lang_key]
            extract = fetch_extract(lang_code, title)
            cleaned = clean_wikipedia_text(extract)

            writer.writerow({
                "qid": entity["qid"],
                "language": lang_code,
                "title": title,
                "extract": cleaned
            })

            time.sleep(1.5)  # Avoid hitting rate limits

## Add categories

In [ ]:
# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd
import requests
from tqdm import tqdm
import time

In [ ]:
file_path = '/content/drive/My Drive/wikipedia/wiki_extracts.csv'

wiki = pd.read_csv(file_path)

wiki.head()

In [ ]:
# Function to fetch instance-of QIDs (P31)
def get_instance_of(qid):
    url = "https://www.wikidata.org/w/api.php"
    params = {
        "action": "wbgetclaims",
        "entity": qid,
        "property": "P31",  # instance of
        "format": "json"
    }
    headers = {
        "User-Agent": "WikiTopicsFetcher/1.0 (mailto:huan.liu2@uzh.ch)"
    }

    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        claims = data.get("claims", {}).get("P31", [])
        topic_ids = [claim['mainsnak']['datavalue']['value']['id']
                     for claim in claims if 'datavalue' in claim['mainsnak']]
        return topic_ids
    except Exception as e:
        return [f"ERROR: {e}"]

In [ ]:
def get_labels_for_qids(qid_list, lang='en'):
    url = "https://www.wikidata.org/w/api.php"
    headers = {
        "User-Agent": "WikiTopicsFetcher/1.0 (mailto:huan.liu2@uzh.ch)"
    }
    label_map = {}

    # Split into batches of 50
    for i in range(0, len(qid_list), 50):
        batch = qid_list[i:i+50]
        params = {
            "action": "wbgetentities",
            "ids": "|".join(batch),
            "props": "labels",
            "languages": lang,
            "format": "json"
        }

        try:
            response = requests.get(url, params=params, headers=headers)
            response.raise_for_status()
            data = response.json()

            for qid in batch:
                label = data.get("entities", {}).get(qid, {}).get("labels", {}).get(lang, {}).get("value")
                if label:
                    label_map[qid] = label
        except Exception as e:
            print(f"Error fetching batch {i//50 + 1}: {e}")
            time.sleep(1)

        time.sleep(0.5)  # avoid being rate-limited

    return label_map

In [ ]:
# test whether this functions works
all_instance_qids = ['Q3624078', 'Q43702', 'Q6256', 'Q20181813', 'Q1250464']
labels = get_labels_for_qids(all_instance_qids, lang='en')
print(labels)

In [ ]:
# Step 1: Fetch all instance-of QIDs
qid_to_instance = {}
for qid in tqdm(wiki['qid'].unique(), desc="Fetching instance-of QIDs"):
    qid_to_instance[qid] = get_instance_of(qid)

# Step 2: Collect all unique instance QIDs
all_instance_qids = set(qid for qids in qid_to_instance.values() for qid in qids if not qid.startswith("ERROR"))

# Step 3: Map instance QIDs to human-readable labels
qid_to_label = get_labels_for_qids(list(all_instance_qids))

# Step 4: Map instance QIDs to labels for each row
def label_list(qids):
    return [qid_to_label.get(qid, qid) for qid in qids]

wiki['semantic_qids'] = wiki['qid'].map(qid_to_instance)
wiki['semantic_labels'] = wiki['semantic_qids'].apply(lambda qids: label_list(qids) if isinstance(qids, list) else qids)

In [ ]:
# Step 5: Save
output_path = "/content/drive/My Drive/wikipedia/wiki_extracts_with_labels.csv"
wiki.to_csv(output_path, index=False)

In [ ]:
wiki_data = pd.read_csv(output_path)
wiki_data.head()

## If more entities are needed, run the following code
Load the previous QIDs and skip any already-seen QIDs to avoid duplicates
In the following script, another 1000 entities are added

In [ ]:
import requests
import bz2
import json
import pandas as pd
import os
import time
import csv
import re
from tqdm import tqdm

In [ ]:
# Setup
url = "https://dumps.wikimedia.org/wikidatawiki/entities/latest-all.json.bz2"
lang_keys = ['enwiki', 'dewiki', 'eswiki', 'zhwiki', 'hiwiki']
entities = []
count = 0
TARGET_COUNT = 1000
buffer = b""

# Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')

# Path to previous CSV
csv_path = "/content/drive/My Drive/wikipedia/wiki_extracts.csv"

# Load previously seen QIDs
previous_qids = set()
if os.path.exists(csv_path):
    df_prev = pd.read_csv(csv_path)
    previous_qids = set(df_prev['qid'].unique())

print(f"Loaded {len(previous_qids)} previously seen QIDs.")

# Start streaming and decompressing
response = requests.get(url, stream=True)
decompressed = bz2.BZ2Decompressor()

for chunk in response.iter_content(4096):
    buffer += decompressed.decompress(chunk)

    while b"\n" in buffer:
        line, buffer = buffer.split(b"\n", 1)
        try:
            line = line.decode("utf-8").strip().rstrip(',')
            if line.startswith('{'):
                obj = json.loads(line)
                qid = obj["id"]
                sitelinks = obj.get("sitelinks", {})

                # Skip already collected QIDs
                if qid in previous_qids:
                    continue

                # Only include if all languages are present
                if all(lang in sitelinks for lang in lang_keys):
                    entity = {
                        "qid": qid,
                        **{lang: sitelinks[lang]["title"] for lang in lang_keys}
                    }
                    entities.append(entity)
                    count += 1
                    if count >= TARGET_COUNT:
                        break
        except Exception:
            continue

    if count >= TARGET_COUNT:
        break

print(f"✅ Collected {count} new unique entities.")

In [ ]:
# File setup
output_file = '/content/drive/My Drive/wikipedia/wiki_extracts.csv'
fieldnames = ['qid', 'language', 'title', 'extract']

# Wikipedia API fetch function
def fetch_extract(lang_code, title):
    url = f"https://{lang_code}.wikipedia.org/w/api.php"
    params = {
        "action": "query",
        "prop": "extracts",
        "format": "json",
        "titles": title,
        "explaintext": 1
    }
    headers = {
        "User-Agent": "WikiArticlesExtractor/1.0 (mailto:huan.liu2@uzh.ch)"
    }
    try:
        response = requests.get(url, params=params, headers=headers)
        response.raise_for_status()
        data = response.json()
        page = next(iter(data["query"]["pages"].values()))
        return page.get("extract", "")
    except Exception as e:
        return f"ERROR: {e}"

In [ ]:
def clean_wikipedia_text(text):
    # Remove section headers
    text = re.sub(r'==+[^=]+==+', '', text)

    # Remove excessive whitespace
    text = re.sub(r'\s+', ' ', text)

    # Stop at the References or See also section
    for section in ['References', 'See also', 'External links', 'Footnotes', 'Bibliography']:
        text = re.split(rf'\b{section}\b', text, flags=re.IGNORECASE)[0]

    return text.strip()

In [ ]:
# Write to CSV
with open(output_file, mode='a', encoding='utf-8', newline='') as file: # mode='a' to append new data into the original csv file instead of overwriting it
    writer = csv.DictWriter(file, fieldnames=fieldnames)
    writer.writeheader()

    for entity in tqdm(entities, desc="Processing entities"):
        for lang_key in ['enwiki', 'dewiki', 'eswiki', 'zhwiki', 'hiwiki']:
            lang_code = lang_key[:2]
            title = entity[lang_key]
            extract = fetch_extract(lang_code, title)
            cleaned = clean_wikipedia_text(extract)

            writer.writerow({
                "qid": entity["qid"],
                "language": lang_code,
                "title": title,
                "extract": cleaned
            })

            time.sleep(1.5)  # Avoid hitting rate limits